In [2]:
import pandas as pd
import unicodedata
import re

df_tm = pd.read_csv("C://Data//base_chile_transfermarkt_final.csv")
df_agg = pd.read_csv("C://Data//chile_players_2025_agg.csv")

In [3]:
def clean_name(name):
    name = str(name).lower()
    name = unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode('utf-8')
    name = re.sub(r'[^a-z\s]', '', name)
    name = re.sub(r'\s+', ' ', name).strip()
    return name

df_tm["name_clean"] = df_tm["Jugador"].apply(clean_name)
df_agg["name_clean"] = df_agg["player_name"].apply(clean_name)

In [4]:
df_agg_unique = df_agg.drop_duplicates(subset="name_clean")

In [5]:
df_merge = df_tm.merge(
    df_agg_unique,
    on="name_clean",
    how="left"
)

In [6]:
match_count = df_merge["player_name"].notna().sum()
total_players = len(df_tm)

print("Matches:", match_count)
print("Total jugadores TM:", total_players)
print("Match %:", round(match_count / total_players * 100, 2))

Matches: 635
Total jugadores TM: 730
Match %: 86.99


In [7]:
df_merge

,Equipo,Liga,Jugador,Edad,Valor,name_clean,competition_name,unique_tournament_id,unique_tournament_name,season_id,...,impacto_negativo,participacion_juego,progresion_de_juego,generador_ocasiones,perdida_de_balon,calidad_de_remate,participacion_ofensiva,frecuencia_fuera_de_juego,intencion_creativa,conversion_remate
0,Deportes Santa Cruz,Primera B,Maximiliano Henríquez,20,0,maximiliano henriquez,Liga de Ascenso (Primera B),1240.0,Liga de Ascenso,71732.0,...,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000
1,Deportes Santa Cruz,Primera B,Braian Camisassa,28,0,braian camisassa,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Deportes Santa Cruz,Primera B,Felipe Alvarado,26,125000,felipe alvarado,Liga de Ascenso (Primera B),1240.0,Liga de Ascenso,71732.0,...,0.0,257.0,0.375000,3.0,0.202614,0.000000,155.0,0.000000,0.019231,0.000000
3,Deportes Santa Cruz,Primera B,Icker Quiroz,19,25000,icker quiroz,Liga de Ascenso (Primera B),1240.0,Liga de Ascenso,71732.0,...,1.0,163.0,0.375000,0.0,0.313131,0.000000,99.0,0.000000,0.000000,0.000000
4,Deportes Santa Cruz,Primera B,David Tati,23,0,david tati,Segunda División,18834.0,Segunda División,68205.0,...,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
725,Ohiggins,Primera,Joaquín Tapia,21,250000,joaquin tapia,Primera División,11653.0,Liga de Primera,71131.0,...,0.0,106.0,0.625000,2.0,0.242424,0.000000,68.0,0.000000,0.050000,0.000000
726,Ohiggins,Primera,Esteban Calderón,21,700000,esteban calderon,Primera División,11653.0,Liga de Primera,71131.0,...,0.0,424.0,0.565517,12.0,0.365591,0.333333,291.0,0.007168,0.082759,0.166667
727,Ohiggins,Primera,Esteban Moreira,23,300000,esteban moreira,Primera División,11653.0,Liga de Primera,71131.0,...,0.0,411.0,0.370370,13.0,0.467391,0.352941,288.0,0.036232,0.088889,0.058824
728,Ohiggins,Primera,Arnaldo Castillo,28,250000,arnaldo castillo,Primera División,11653.0,Liga de Primera,71131.0,...,0.0,539.0,0.442211,11.0,0.438235,0.304348,349.0,0.017647,0.045226,0.130435


In [8]:
# Guardar CSV
df_merge.to_csv(r"C:\Data\df_merge.csv", index=False)

print("Archivo guardado en C:\\Data\\df_merge.csv")

Archivo guardado en C:\Data\df_merge.csv


In [9]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
df_consolidado_final = df_merge[df_merge["player_name"].notna()].copy()

no_metricas = [
    'player_name', 'team_name', 'competition_name', 
    'unique_tournament_name', 'player_short_name', 
    'player_slug', 'position', 'Valor', 'Edad',
    'unique_tournament_id', 'season_id', 
    'team_id', 'player_id', 'shirt_number',
    'Equipo', 'name_clean'
]

In [13]:
metricas_cols = [
    c for c in df_consolidado_final.columns
    if c not in no_metricas
    and np.issubdtype(df_consolidado_final[c].dtype, np.number)
]

In [18]:
df_sim = df_consolidado_final.copy()

df_sim = df_sim.reset_index(drop=True)  # 🔥 CLAVE

df_sim[metricas_cols] = df_sim[metricas_cols].fillna(0)

scaler = StandardScaler()
scaled_features = scaler.fit_transform(df_sim[metricas_cols])

df_sim['player_name'] = df_sim['player_name'].str.strip()

In [47]:
def buscador_scouting_final(nombre_buscado):
    try:
        jugador_match = df_sim[df_sim['player_name'].str.contains(nombre_buscado, case=False)]
        
        if jugador_match.empty:
            print(f"❌ No se encontró a '{nombre_buscado}'.")
            return
        
        idx = jugador_match.index[0]
        jugador_foco = df_sim.iloc[idx]
        
        # FILTRO DE PARES
        df_pares = df_sim[
        (df_sim['position'] == jugador_foco['position']) & 
        (df_sim['Edad'].between(jugador_foco['Edad']-3, jugador_foco['Edad']+3)) &
        (df_sim['minutes_played_total'] >= 450)   # 🔥 filtro clave
        ].copy()
        indices_pares = df_pares.index
        
        if len(indices_pares) < 6:
            print("⚠️ Muy pocos jugadores comparables.")
            return
        
        idx_relativo = list(indices_pares).index(idx)
        scaled_pares = scaled_features[indices_pares]
        
        distancias = cosine_similarity(
            [scaled_pares[idx_relativo]],
            scaled_pares
        )
        
        df_pares['Similitud_%'] = distancias[0] * 100
        
        indices_top = distancias[0].argsort()[-6:-1][::-1]
        gemelos = df_pares.iloc[indices_top].copy()
        
        gemelos_validos = gemelos[gemelos['Valor'] > 0].copy()
        
        es_exportable = (jugador_foco['Edad'] <= 24) and (jugador_foco['Valor'] >= 750000)
        
        print("="*60)
        print(f"📊 INFORME DE MERCADO: {jugador_foco['player_name'].upper()}")
        print(f"⚽ Club 2026: {jugador_foco['Equipo']}")
        print(f"📅 Club 2025: {jugador_foco['team_name']}")
        print(f"🎂 Edad: {int(jugador_foco['Edad'])} años")
        print(f"🛡️ Posición: {jugador_foco['position']}")
        print("="*60)
        print(f"💰 Valor Actual: {jugador_foco['Valor']:,.0f} €")
        
        if len(gemelos_validos) > 0:

                                            # 🔥 Pesos combinados: similitud + minutos
            pesos = (
            gemelos_validos['Similitud_%'] *
            gemelos_validos['minutes_played_total']
            )

            fv = np.average(
                gemelos_validos['Valor'],
                weights=pesos
            )

            diferencia = fv - jugador_foco['Valor']
            
            if es_exportable:
                status = "✈️ PROYECTO DE EXPORTACIÓN"
            elif diferencia > 150000:
                status = "🟢 INFRAVALORADO"
            elif diferencia < -150000:
                status = "🔴 SOBREVALORADO"
            else:
                status = "🟡 PRECIO JUSTO"
                
            print(f"⚖️ Fair Value: {fv:,.0f} €")
            print(f"📢 Estatus: {status}")
            print(f"📈 Margen vs Mercado: {diferencia:,.0f} €")
        else:
            print("⚖️ No hay suficientes referencias para estimar Fair Value.")
        
        print("\n👯 GEMELOS ESTADÍSTICOS:")
        
        cols_fin = [
            'player_name', 'team_name', 'Equipo',
            'Edad', 'Valor', 'Similitud_%',
            'rating_avg', 'minutes_played_total'
        ]
        
        gemelos_display = gemelos[cols_fin].copy()
        gemelos_display['Similitud_%'] = gemelos_display['Similitud_%'].map("{:.2f}%".format)
        
        display(gemelos_display)
        print("="*60)
        
    except Exception as e:
        print(f"⚠️ Error: {e}")

In [57]:
buscador_scouting_final("Jeisson Vargas")

📊 INFORME DE MERCADO: JEISSON VARGAS
⚽ Club 2026: Deportes La Serena
📅 Club 2025: Deportes La Serena
🎂 Edad: 28 años
🛡️ Posición: F
💰 Valor Actual: 750,000 €
⚖️ Fair Value: 791,303 €
📢 Estatus: 🟡 PRECIO JUSTO
📈 Margen vs Mercado: 41,303 €

👯 GEMELOS ESTADÍSTICOS:


,player_name,team_name,Equipo,Edad,Valor,Similitud_%,rating_avg,minutes_played_total
133,Rodrigo Contreras,Universidad de Chile,Deportes Antofagasta,30,1200000,61.12%,7.009091,864.0
557,Facundo Pons,Deportes Limache,Deportes Limache,30,450000,52.08%,7.362500,670.0
608,César Munder,Cobresal,Palestino,26,850000,51.99%,7.036364,824.0
406,Diego Valencia,Universidad Católica,Universidad Católica,25,700000,51.70%,6.816667,886.0
318,César González,Deportes Iquique,Deportes Iquique,28,450000,50.99%,6.825000,460.0
